# 470 — Concatenated ROIs on exemplar ERSPs

Validation figure: for a hand-picked list of **information-rich** contacts, build the
**concatenated `[audio | picture | reading]`** ERSP (the triptych — *not* one map per condition)
and **overlay the concatenated role ROIs** (`roi_config_concatenated.py`). The unique
(block × time × frequency) box regions are drawn as outlines coloured by frequency band, with
block dividers and per-block response-onset lines — so you see exactly where the role templates
sit on real cross-condition activity.

Each `NAMES` entry identifies a **contact** (pid + electrode); all three conditions are loaded for
it (contacts missing any condition are skipped). `GRID='full'` uses the native 129×900 triptych.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
NAMES = [
    "EL045_reading_WM_ERSP_pH_L11_TN_CLEAN.png",
    "PAT_3415_reading_WM_ERSP_GE4_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_OPG1_TN_CLEAN.png",
    "PAT_6704_reading_WM_ERSP_TPD5_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IAG7_TN_CLEAN.png",
    "EL043_reading_WM_ERSP_iSMG3_TN_CLEAN.png",
    "EL042_reading_WM_ERSP_STG_R4_TN_CLEAN.png",
    "PAT_3415_reading_WM_ERSP_OS2_TN_CLEAN.png",
    "EL038_reading_WM_ERSP_aI_R8_TN_CLEAN.png",
    "EL033_reading_WM_ERSP_PHG_R12_TN_CLEAN.png",
    "PAT_3390_reading_WM_ERSP_CPG15_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IMG9_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IAG3_TN_CLEAN.png",
    "EL030_reading_WM_ERSP_A_L15_TN_CLEAN.png",
    "PAT_5533_reading_WM_ERSP_OFD5_TN_CLEAN.png",
    "EL045_reading_WM_ERSP_pH_L15_TN_CLEAN.png",
    "EL035_audio_WM_ERSP_TTG_R1_TN_CLEAN.png",
    "EL045_audio_WM_ERSP_aH_L12_TN_CLEAN.png",
]
GRID = 'full'                        # 'full' = native 129x900 triptych
print(len(NAMES), 'names · grid:', GRID)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
run_dir = P.new_run_dir('roi_on_concat')
ok = miss = 0
seen = set()
for name in NAMES:
    cid = P._contact_id_from_name(name)          # (pid, mid) — condition-independent
    if cid is None or cid in seen:
        continue
    seen.add(cid)
    try:
        concat = P.load_concat_ersp(INPUT_DIR, name)     # (129, 3*300) [audio|picture|reading]
    except (FileNotFoundError, ValueError) as e:
        print('  [skip]', e); miss += 1; continue
    label = f'{cid[0]} {cid[1].split("_ERSP_")[-1]}'
    stem = f'{cid[0]}_{cid[1]}'.replace('/', '_')
    P.plot_roles_on_concat(concat, grid=GRID, label=label, out_png=run_dir / f'{stem}.png')
    plt.close('all')
    display(Image(filename=str(run_dir / f'{stem}.png')))
    ok += 1
print(f'\ndone: {ok} contacts plotted, {miss} skipped (missing a condition) · saved -> {run_dir}')


## Per-role exemplars — one matching ERSP per role
For each role, pick a contact that **460 assigned to that role** and show its concatenated ERSP
with **that role's own template** overlaid (pos = red fill, neg = blue fill, zero = grey dashed =
"must be silent"). This is the "here's what an *auditory* / *motor* / … contact looks like" panel.
**Run `460` first** (it writes `role_table_<grid>.parquet`). `GRID_ROLES` must match the grid you
ran 460 with.


In [ ]:
GRID_ROLES   = 'full'                # must match the grid you ran 460 with ('full' or 'ds')
N_PER_ROLE   = 1                     # exemplars per role
try:
    df_role = P.load_role_table(grid=GRID_ROLES)
    print('role table:', df_role.shape, '| assigned roles:',
          df_role[df_role.role != 'none'].role.value_counts().to_dict())
    ex = P.plot_role_exemplars(INPUT_DIR, df_role, grid=GRID_ROLES,
                               n_per_role=N_PER_ROLE, run_dir=run_dir)
    for rn, paths in ex.items():
        for p in paths:
            plt.close('all')
            display(Image(filename=str(p)))
except FileNotFoundError:
    print('no role table yet — run 460 first to produce role_table_%s.parquet' % GRID_ROLES)


## Discretized −1/0/+1 of each exemplar (what the role matching actually sees)
Shows the **score-gated discretized** concatenated map (blue = −1, white = 0, red = +1) — the
representation `box_expresses` runs on. If it looks too sparse, the discretization is too strict:
**lower `SCORE_PCT_VIZ`** (or set `None` for no score gate) and/or **loosen `SEG_KWARGS`** (e.g.
`thr_pos` 2.0→1.5, `thr_neg` −4.0→−3.0, `max_blobs` 4→8) and re-run to see more cells survive.


In [ ]:
SCORE_PCT_VIZ = 33.0          # blob-score gate percentile (lower = looser); None = no gate
SEG_KWARGS    = None          # e.g. {'thr_pos': 1.5, 'thr_neg': -3.0, 'max_blobs': 8} to loosen
if SCORE_PCT_VIZ is None:
    score_min = None
else:
    dfm, Xf = P.prepare_pooling_dataset(INPUT_DIR)
    score_min = P.resolve_score_gate(Xf, pct=SCORE_PCT_VIZ)
print('score gate:', score_min, '| seg overrides:', SEG_KWARGS)
seen2 = set()
for name in NAMES:
    cid = P._contact_id_from_name(name)
    if cid is None or cid in seen2:
        continue
    seen2.add(cid)
    try:
        disc = P.load_concat_discretized(INPUT_DIR, name, score_min=score_min, seg_kwargs=SEG_KWARGS)
    except (FileNotFoundError, ValueError) as e:
        print('  [skip]', e); continue
    stem = f'{cid[0]}_{cid[1]}'.replace('/', '-')
    P.plot_discretized_on_concat(disc, label=f'{cid[0]} {cid[1].split("_ERSP_")[-1]}',
                                 out_png=run_dir / f'disc_{stem}.png')
    plt.close('all')
    display(Image(filename=str(run_dir / f'disc_{stem}.png')))
print('done')
